Task 1: GPT-2 Fine-Tuning for Text Generation

Model: GPT-2 (124M parameters)
Framework: HuggingFace Transformers
Training Environment: Google Colab (GPU enabled)
Dataset: Custom AI-in-Education corpus
Epochs: 10 (initial), retrained on expanded dataset
Objective: Fine-tune GPT-2 to generate contextually relevant text on AI in Indian education.

In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
!pip install transformers
!pip install torch

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

In [ ]:
prompt = "Artificial Intelligence will transform education in India by"

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=100,
    pad_token_id=tokenizer.eos_token_id
)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generated_text)

Artificial Intelligence will transform education in India by providing a new way to teach and learn.

The Indian government has been working on a new way to teach AI in schools for years. The government has been working on a new way to teach AI in schools for years.

The Indian government has been working on a new way to teach AI in schools for years.

The Indian government has been working on a new way to teach AI in schools for years.

The Indian government


In [ ]:
prompt = "Artificial Intelligence will transform education in India by"

inputs = tokenizer(prompt, return_tensors="pt")

beam_outputs = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=100,
    num_beams=5,
    early_stopping=True,
    pad_token_id=tokenizer.eos_token_id
)

print("Beam Search Output:\n")
print(tokenizer.decode(beam_outputs[0], skip_special_tokens=True))

Beam Search Output:

Artificial Intelligence will transform education in India by making it easier for students to learn about the world around them.

India is one of the fastest-growing countries in the world, with a population of more than 1.5 billion people. The country is also one of the fastest-growing countries in the world, with a population of more than 1.5 billion people. The country is also one of the fastest-growing countries in the world, with a population of more than 1.5


In [ ]:
prompt = "Artificial Intelligence will transform education in India by"

inputs = tokenizer(prompt, return_tensors="pt")

topk_outputs = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=100,
    do_sample=True,
    top_k=50,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id
)

print("Top-K Sampling Output:\n")
print(tokenizer.decode(topk_outputs[0], skip_special_tokens=True))

Top-K Sampling Output:

Artificial Intelligence will transform education in India by 2025 into a global enterprise.

Niraj Mahadevan, head of innovation and research at the Foundation for Strategic, Community, and Technical Research, said the government is committed to bringing AI into the next stage. "There will be more data collected, more quality control, more technical education, more knowledge, more job creation; we will also create a new kind of business."


In [ ]:
prompt = "Artificial Intelligence will transform education in India by"

inputs = tokenizer(prompt, return_tensors="pt")

topp_outputs = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=100,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id
)

print("Top-P (Nucleus) Sampling Output:\n")
print(tokenizer.decode(topp_outputs[0], skip_special_tokens=True))

Top-P (Nucleus) Sampling Output:

Artificial Intelligence will transform education in India by empowering the young and the elderly to take charge of their own lives.

The first step is to create a world where young people will be able to do their own research. This is what a "Smart Research" programme is all about. The aim is to create a world where young people will have the opportunity to take charge of their own lives, making their own choices.

The smart research programme is designed to help young people make better informed


In [ ]:
!pip install datasets accelerate

In [ ]:
text = """
Artificial Intelligence is transforming education in India.
AI-powered classrooms personalize learning for every student.
Machine learning helps teachers track student performance.
Smart education platforms improve accessibility in rural areas.
The future of Indian education will be driven by AI innovation.
AI tools help automate grading and reduce teacher workload.
Digital learning powered by AI increases student engagement.
Government initiatives are promoting AI literacy in schools.
AI-based tutoring systems provide customized learning paths.
Data-driven education will shape the next generation of learners.
"""

with open("dataset.txt", "w") as f:
    f.write(text)

print("Dataset file created successfully!")

Dataset file created successfully!


In [ ]:
from datasets import load_dataset
from transformers import GPT2Tokenizer

# Load tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Load dataset
dataset = load_dataset("text", data_files={"train": "dataset.txt"})

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

print("Tokenization completed!")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

Tokenization completed!


In [ ]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained("gpt2")

model.resize_token_embeddings(len(tokenizer))

print("GPT-2 model loaded successfully!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT-2 model loaded successfully!


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    logging_steps=10,
    save_strategy="no",
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
)

print("Trainer setup completed!")

Trainer setup completed!


In [ ]:
def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize_function, batched=True)

print("Tokenization updated with labels!")

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

Tokenization updated with labels!


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    logging_steps=10,
    save_strategy="no",
    report_to=[],
    remove_unused_columns=False   # ⭐ THIS FIXES YOUR ERROR
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
)

print("Trainer reconfigured successfully!")

Trainer reconfigured successfully!


In [ ]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,2.487351
20,0.756486
30,0.161236
40,0.147910
50,0.097974
60,0.089827


TrainOutput(global_step=60, training_loss=0.6234640767176945, metrics={'train_runtime': 7.646, 'train_samples_per_second': 14.387, 'train_steps_per_second': 7.847, 'total_flos': 7185530880000.0, 'train_loss': 0.6234640767176945, 'epoch': 10.0})

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

output = generator(
    "Artificial Intelligence in Indian schools",
    max_length=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True
)

print(output[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'max_length', 'top_p', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Artificial Intelligence in Indian schools has helped improve performance of teachers.


In [ ]:
large_text = """
Artificial Intelligence is transforming education in India.
AI-powered classrooms personalize learning experiences.
Machine learning helps teachers analyze student performance.
Smart education platforms improve accessibility in rural areas.
Government initiatives promote AI literacy among students.
Digital tools reduce administrative workload for teachers.
AI-based tutoring systems provide customized lesson plans.
Data-driven insights help schools make better decisions.
Online AI platforms support remote learning effectively.
Adaptive learning systems improve student engagement.

Artificial Intelligence enhances skill development in engineering colleges.
AI labs are being introduced in Indian universities.
Students are learning Python, machine learning, and robotics.
Virtual assistants help clarify doubts instantly.
AI encourages innovation and research culture.
Smart grading systems reduce bias in evaluation.
Educational chatbots provide 24/7 academic support.
AI-based analytics predict dropout risks.
Digital transformation is reshaping traditional classrooms.
AI makes education more inclusive and accessible.
""" * 10   # Repeat to expand dataset

with open("dataset.txt", "w") as f:
    f.write(large_text)

print("Large dataset created successfully!")

Large dataset created successfully!


In [ ]:
def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = load_dataset("text", data_files={"train": "dataset.txt"})
tokenized_dataset = dataset.map(tokenize_function, batched=True)

print("Re-tokenization completed!")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/220 [00:00<?, ? examples/s]

Re-tokenization completed!


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    logging_steps=20,
    save_strategy="no",
    report_to=[],
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
)

trainer.train()

Step,Training Loss
20,0.166993
40,0.073071
60,0.041255
80,0.030198
100,0.030068
120,0.025044
140,0.023633
160,0.026126


TrainOutput(global_step=165, training_loss=0.05149594816294584, metrics={'train_runtime': 29.0497, 'train_samples_per_second': 22.72, 'train_steps_per_second': 5.68, 'total_flos': 43113185280000.0, 'train_loss': 0.05149594816294584, 'epoch': 3.0})

In [ ]:
base_model = GPT2LMHeadModel.from_pretrained("gpt2")
base_generator = pipeline("text-generation", model=base_model, tokenizer=tokenizer)

print("BEFORE FINE-TUNING:\n")
print(base_generator("AI in Indian education", max_new_tokens=60)[0]["generated_text"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BEFORE FINE-TUNING:

AI in Indian education. I was a student in an English-language school. I was interested in the nature of mathematics and my first objective was to understand the origin of mathematics. I was motivated by the fact that I had been taught to appreciate the concept of general relativity. Therefore I wanted to understand the role of the


In [ ]:
print("\nAFTER FINE-TUNING:\n")
print(generator("AI in Indian education", max_new_tokens=60, do_sample=True)[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



AFTER FINE-TUNING:

AI in Indian education systems.


In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print(generator(
    "AI in Indian education",
    max_new_tokens=60,
    do_sample=True,
    temperature=0.8,
    top_p=0.9
)[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'top_p', 'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AI in Indian education.
